<a href="https://colab.research.google.com/github/roggersanguzu/Intrusion-detection-ML-Models/blob/main/Intrusion_detectionXGBOOST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import matplotlib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder,
    FunctionTransformer
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 150)
sns.set_style('darkgrid')
plt.rcParams['font.size'] = 14
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['patch.edgecolor'] = '#313338'

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
monday=pd.read_csv("/content/drive/MyDrive/Machine Learning Projects/Monday-WorkingHours.pcap_ISCX.csv")
tuesday=pd.read_csv("/content/drive/MyDrive/Machine Learning Projects/Tuesday-WorkingHours.pcap_ISCX.csv")
wednesday=pd.read_csv("/content/drive/MyDrive/Machine Learning Projects/Wednesday-workingHours.pcap_ISCX.csv")
thursday_web=pd.read_csv("/content/drive/MyDrive/Machine Learning Projects/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv")
thursday_infilteration=pd.read_csv("/content/drive/MyDrive/Machine Learning Projects/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv")
friday_morning=pd.read_csv("/content/drive/MyDrive/Machine Learning Projects/Friday-WorkingHours-Morning.pcap_ISCX.csv")
friday_portscan=pd.read_csv("/content/drive/MyDrive/Machine Learning Projects/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
friday_DDos=pd.read_csv("/content/drive/MyDrive/Machine Learning Projects/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")

Mounted at /content/drive


In [ ]:
monday["source"] = "Monday"
tuesday["source"] = "Tuesday"
wednesday["source"] = "Wednesday"
thursday_web["source"] = "Thursday-Web"
thursday_infilteration["source"] = "Thursday-Infiltration"
friday_morning["source"] = "Friday-Morning"
friday_portscan["source"] = "Friday-PortScan"
friday_DDos["source"] = "Friday-DDoS"

In [ ]:
dfs = [monday, tuesday, wednesday, thursday_web,
       thursday_infilteration, friday_morning,
       friday_portscan, friday_DDos]
for df in dfs:
    df.columns = df.columns.str.strip()

In [ ]:
for name, df in zip(
    ["Monday","Tuesday","Wednesday","Thursday-Web",
     "Thursday-Infiltration","Friday-Morning",
     "Friday-PortScan","Friday-DDoS"],
    dfs
):
    print("\n", name)
    print(df.describe())


 Monday
       Destination Port  Flow Duration  Total Fwd Packets  \
count     529918.000000   5.299180e+05      529918.000000   
mean       10644.367112   1.038927e+07          10.390315   
std        21390.213475   2.875195e+07         892.412791   
min            0.000000  -1.000000e+00           1.000000   
25%           53.000000   1.760000e+02           2.000000   
50%           80.000000   3.130300e+04           2.000000   
75%          443.000000   3.557448e+05           4.000000   
max        65535.000000   1.200000e+08      219759.000000   

       Total Backward Packets  Total Length of Fwd Packets  \
count           529918.000000                 5.299180e+05   
mean                11.517105                 5.324195e+02   
std               1173.318788                 6.228642e+03   
min                  0.000000                 0.000000e+00   
25%                  1.000000                 1.800000e+01   
50%                  2.000000                 6.800000e+01   
75%    

In [ ]:
full_df = pd.concat(dfs, ignore_index=True)
full_df = full_df.replace([np.inf, -np.inf], np.nan).dropna()

In [ ]:
full_df = full_df.replace([np.inf, -np.inf], np.nan)
full_df = full_df.drop_duplicates()
full_df = full_df.dropna()

In [ ]:
full_df['Label'] = full_df['Label'].astype(str).str.strip()

In [ ]:
le = LabelEncoder()
y = le.fit_transform(full_df['Label'])

In [ ]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(label_mapping)

{'BENIGN': np.int64(0), 'Bot': np.int64(1), 'DDoS': np.int64(2), 'DoS GoldenEye': np.int64(3), 'DoS Hulk': np.int64(4), 'DoS Slowhttptest': np.int64(5), 'DoS slowloris': np.int64(6), 'FTP-Patator': np.int64(7), 'Heartbleed': np.int64(8), 'Infiltration': np.int64(9), 'PortScan': np.int64(10), 'SSH-Patator': np.int64(11), 'Web Attack � Brute Force': np.int64(12), 'Web Attack � Sql Injection': np.int64(13), 'Web Attack � XSS': np.int64(14)}


In [ ]:
X = full_df.drop(['Label', 'source'], axis=1)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
import numpy as np
from sklearn.preprocessing import FunctionTransformer

def feature_engineering(X):
    X = X.copy()
    X['bytes_per_packet'] = (X['Total Length of Fwd Packets'] + X['Total Length of Bwd Packets']) / (X['Total Fwd Packets'] + 1)
    X['packet_burst'] = X['Flow Packets/s'] * X['Flow Bytes/s']
    X['flow_asymmetry'] = X['Total Fwd Packets'] / (X['Total Backward Packets'] + 1)
    X['iat_instability'] = X['Flow IAT Std'] / (X['Flow IAT Mean'] + 1)
    return X
feature_transformer = FunctionTransformer(feature_engineering)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

In [ ]:
pipeline = ImbPipeline(steps=[
    ('feature_engineering', feature_transformer),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', xgb_model)
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

% ReductionOfDataSets

In [ ]:
X_dev = X.sample(frac=0.2, random_state=42)
y_series = pd.Series(y, index=X.index)
y_dev = y_series[X_dev.index]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_dev,
    y_dev,
    test_size=0.2,
    random_state=42,
    stratify=y_dev
)

In [ ]:
import numpy as np
from sklearn.preprocessing import FunctionTransformer
def feature_engineering(X):
    X = X.copy()
    X['bytes_per_packet'] = (X['Total Length of Fwd Packets'] + X['Total Length of Bwd Packets']) / (X['Total Fwd Packets'] + 1)
    X['packet_burst'] = X['Flow Packets/s'] * X['Flow Bytes/s']
    X['flow_asymmetry'] = X['Total Fwd Packets'] / (X['Total Backward Packets'] + 1)
    X['iat_instability'] = X['Flow IAT Std'] / (X['Flow IAT Mean'] + 1)
    return X
feature_transformer = FunctionTransformer(feature_engineering)

In [ ]:
pipeline = ImbPipeline(steps=[
    ('feature_engineering', feature_transformer),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', xgb_model)
])

In [ ]:
class_counts = pd.Series(y_train).value_counts()
problematic_classes = class_counts[class_counts < 6].index
non_problematic_indices = pd.Series(y_train).index[~pd.Series(y_train).isin(problematic_classes)]

X_train = X_train.loc[non_problematic_indices]
y_train = y_train.loc[non_problematic_indices]

print(f"Removed classes with fewer than 6 samples: {list(problematic_classes.values)}")
print(f"New X_train shape: {X_train.shape}")
print(f"New y_train shape: {y_train.shape}")

pipeline.fit(X_train, y_train)

Removed classes with fewer than 6 samples: [np.int64(13), np.int64(8)]
New X_train shape: (411615, 78)
New y_train shape: (411615,)


In [ ]:
from sklearn.metrics import classification_report, accuracy_score
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))